# 03 — Gold: dim_carrier

| Property | Value |
|----------|-------|
| **Gold Table** | `dim_carrier` |
| **Grain** | One row per GlobalCarrierId |
| **Source** | `ref.CarrierHierarchy` (superset of `rpt.vwCarrierHierarchy`) |
| **PK** | `GlobalCarrierId` (int) |
| **Rows** | 37,180 |

**Hierarchy**: GlobalParent → OperatingCompany → CarrierName

**Join**: `dim_party.CompCode = dim_carrier.CompCode` (for carrier-type parties)

In [ ]:
# ============================================================
# Cell 1: Setup & Config
# ============================================================
from pyspark.sql import functions as F

LAKEHOUSE = "The_Global_Loom"
TABLE = "dim_carrier"
SOURCE_TABLE = "ref.CarrierHierarchy"

print(f"✅ Config: {SOURCE_TABLE} → {LAKEHOUSE}.{TABLE}")

In [ ]:
# ============================================================
# Cell 2: Read silver source
# ============================================================
df_src = spark.table(SOURCE_TABLE)

print(f"📥 Source: {df_src.count():,} rows × {len(df_src.columns)} cols")
df_src.printSchema()

## Cell 3: Transform

- Keep hierarchy columns: GlobalParent → OperatingCompany → CarrierName
- CompCode is the join key to dim_party (unique)
- Drop: ETL dates

In [ ]:
# ============================================================
# Cell 3: Transform
# ============================================================
df_clean = df_src.select(
    F.col("GlobalCarrierId").cast("int"),
    F.col("CompCode").cast("string"),
    F.col("CarrierName").cast("string"),
    F.col("OperatingCompanyId").cast("int"),
    F.col("OperatingCompany").cast("string"),
    F.col("GlobalParentId").cast("int"),
    F.col("GlobalParent").cast("string"),
    F.col("CountryCode").cast("string"),
    F.col("IsDeleted").cast("boolean")
)

print(f"✅ After column select: {df_clean.count():,} rows × {len(df_clean.columns)} cols")

In [ ]:
# ============================================================
# Cell 4: Add Unknown member
# ============================================================
unknown_row = spark.createDataFrame([(
    -1, "Unknown", "Unknown", -1, "Unknown",
    -1, "Unknown", "Unknown", False
)], schema=df_clean.schema)

df_final = df_clean.unionByName(unknown_row)

print(f"✅ Added Unknown member: {df_final.count():,} rows")

In [ ]:
# ============================================================
# Cell 5: Data quality checks
# ============================================================
total = df_final.count()
dupes_carrier = total - df_final.select("GlobalCarrierId").distinct().count()
dupes_comp = total - df_final.select("CompCode").distinct().count()

print(f"✅ DQ Checks")
print(f"   Total rows:             {total:,}")
print(f"   Duplicate CarrierIds:   {dupes_carrier}")
print(f"   Duplicate CompCodes:    {dupes_comp}")

assert dupes_comp == 0, f"❌ CompCode has {dupes_comp} duplicates!"
print("\n✅ All DQ checks passed")

In [ ]:
# ============================================================
# Cell 6: Write to gold lakehouse
# ============================================================
df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{LAKEHOUSE}.{TABLE}")

print(f"✅ Written: {LAKEHOUSE}.{TABLE}")
print(f"   Rows: {spark.table(f'{LAKEHOUSE}.{TABLE}').count():,}")